# ASIC Static Data: First Glance and Harmonization Draft

This notebook recreates the first-pass ASIC static exploration and applies the current harmonization rules.

Current scope:
- load the static files for all hospitals,
- compare raw schemas,
- harmonize to a shared static schema,
- rename `patient_id` to `stay_id`,
- recode `sex` from `W/M` to `F/M`,
- derive binary in-hospital and in-ICU mortality columns,
- turn `-1` sentinel values into missing where appropriate,
- group the final columns by category.

Note:
- The requested code mapping `L/M/P/1/2/3/X` appears in the raw `BMI` column, not in the raw weight bucket column. In this notebook, that mapping is therefore applied to `bmi_group`.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.width", 220)

RAW_DATA_DIR = Path("/Users/joanameyer/data/asic/raw_sample10")
STATIC_GLOB = "asic_*_static.csv"

static_files = sorted(RAW_DATA_DIR.glob(STATIC_GLOB))
if not static_files:
    raise FileNotFoundError(f"No files matching {STATIC_GLOB!r} found in {RAW_DATA_DIR}")

static_files


[PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK00_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK02_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK03_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK04_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK06_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK07_static.csv'),
 PosixPath('/Users/joanameyer/data/asic/raw_sample10/asic_UK08_static.csv')]

In [2]:
def hospital_name_from_path(path: Path) -> str:
    return path.stem.replace("_static", "")


def normalize_missing(series: pd.Series) -> pd.Series:
    if pd.api.types.is_object_dtype(series) or pd.api.types.is_string_dtype(series):
        return series.replace(r"^\s*$", pd.NA, regex=True)
    return series


def to_clean_string(series: pd.Series) -> pd.Series:
    return normalize_missing(series).astype("string").str.strip()


def series_values_equal(left: pd.Series, right: pd.Series) -> bool:
    left_norm = to_clean_string(left).fillna("<NA>")
    right_norm = to_clean_string(right).fillna("<NA>")
    return left_norm.equals(right_norm)


def replace_minus_one_with_na(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return series.mask(series == -1)
    cleaned = to_clean_string(series)
    return cleaned.replace({"-1": pd.NA, "-1.0": pd.NA})


def empty_columns(df: pd.DataFrame) -> list[str]:
    return [column for column in df.columns if normalize_missing(df[column]).isna().all()]


def example_values(series: pd.Series, max_values: int = 8) -> list[str]:
    normalized = normalize_missing(series)
    unique_values = normalized.dropna().astype(str).unique().tolist()
    preview = unique_values[:max_values]
    if len(unique_values) > max_values:
        preview.append("...")
    return preview


def value_preview(df: pd.DataFrame, max_values: int = 8) -> pd.DataFrame:
    rows = []
    for column in df.columns:
        normalized = normalize_missing(df[column])
        non_null = normalized.dropna()
        rows.append(
            {
                "column": column,
                "dtype": str(df[column].dtype),
                "non_null_count": int(non_null.shape[0]),
                "n_unique_non_null": int(non_null.nunique()),
                "example_values": example_values(df[column], max_values=max_values),
            }
        )
    return pd.DataFrame(rows)


## Raw Overview

Load all hospital static files exactly as they are stored.


In [3]:
raw_static_tables = {
    hospital_name_from_path(path): pd.read_csv(path)
    for path in static_files
}

overview_rows = []
for hospital, df in raw_static_tables.items():
    overview_rows.append(
        {
            "hospital": hospital,
            "rows": df.shape[0],
            "raw_columns": df.shape[1],
            "empty_raw_columns": len(empty_columns(df)),
        }
    )

display(pd.DataFrame(overview_rows).sort_values("hospital").reset_index(drop=True))


,hospital,rows,raw_columns,empty_raw_columns
0,asic_UK00,10,18,5
1,asic_UK02,10,18,0
2,asic_UK03,10,13,0
3,asic_UK04,10,16,0
4,asic_UK06,10,18,0
5,asic_UK07,10,18,0
6,asic_UK08,10,18,0


In [4]:
raw_column_orders = {hospital: list(df.columns) for hospital, df in raw_static_tables.items()}
raw_column_sets = {hospital: set(df.columns) for hospital, df in raw_static_tables.items()}

reference_hospital = sorted(raw_static_tables)[0]
reference_columns = raw_column_orders[reference_hospital]
reference_set = set(reference_columns)

comparison_rows = []
for hospital in sorted(raw_static_tables):
    current_columns = raw_column_orders[hospital]
    current_set = raw_column_sets[hospital]
    comparison_rows.append(
        {
            "hospital": hospital,
            "same_raw_column_order_as_reference": current_columns == reference_columns,
            "same_raw_column_set_as_reference": current_set == reference_set,
            "missing_vs_reference": sorted(reference_set - current_set),
            "extra_vs_reference": sorted(current_set - reference_set),
        }
    )

display(pd.DataFrame(comparison_rows))
print("Common raw columns:")
print(sorted(set.intersection(*raw_column_sets.values())))


,hospital,same_raw_column_order_as_reference,same_raw_column_set_as_reference,missing_vs_reference,extra_vs_reference
0,asic_UK00,True,True,[],[]
1,asic_UK02,False,True,[],[]
2,asic_UK03,False,False,"[Beatmungsfreie_Tage, Dialyse_(dialysefreie_Tage), Liegedauer_KH, Sterblichkeit, Wiederaufnahme_ICU]",[]
3,asic_UK04,False,False,"[Beatmungsfreie_Tage, Dialyse_(dialysefreie_Tage), clusterAlter, clusterGeschlecht, clusterKoerpergewicht, clusterKoerpergroesse]","[ClusterAlter, ClusterGeschlecht, ClusterKoerperGewicht, ClusterKoerpergroesse]"
4,asic_UK06,True,True,[],[]
5,asic_UK07,False,True,[],[]
6,asic_UK08,False,True,[],[]


Common raw columns:
['BMI', 'Cluster-ID', 'Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)', 'ICD-10_Codes', 'Liegedauer_ICU', 'Phase', 'Pseudo-ID', 'PseudoID', 'Zeit_seit_Studienbeginn']


## Harmonized Schema

The final harmonized columns are ordered by category.


In [5]:
STANDARD_SOURCES = {
    "stay_id": ["PseudoID", "Pseudo-ID"],
    "sex": ["clusterGeschlecht", "ClusterGeschlecht"],
    "age_group": ["clusterAlter", "ClusterAlter"],
    "height_group": ["clusterKoerpergroesse", "ClusterKoerpergroesse"],
    "weight_group": ["clusterKoerpergewicht", "ClusterKoerperGewicht"],
    "bmi_group": ["BMI"],
    "hosp_mortality": ["Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)"],
    "icu_mortality": ["Sterblichkeit"],
    "hosp_los": ["Liegedauer_KH"],
    "icu_los": ["Liegedauer_ICU"],
    "icu_readmit": ["Wiederaufnahme_ICU"],
    "dialysis_free_days": ["Dialyse_(dialysefreie_Tage)"],
    "vent_free_days": ["Beatmungsfreie_Tage"],
    "icd10_codes": ["ICD-10_Codes"],
}

FINAL_COLUMNS = [
    "hospital_id",
    "stay_id",
    "age_group",
    "sex",
    "height_group",
    "weight_group",
    "bmi_group",
    "hosp_mortality",
    "icu_mortality",
    "hosp_los",
    "icu_los",
    "icu_readmit",
    "dialysis_free_days",
    "vent_free_days",
    "icd10_codes",
]

NUMERIC_COLUMNS = [
    "stay_id",
    "hosp_mortality",
    "icu_mortality",
    "hosp_los",
    "icu_los",
    "icu_readmit",
    "dialysis_free_days",
    "vent_free_days",
]

STRING_COLUMNS = [
    "hospital_id",
    "age_group",
    "sex",
    "height_group",
    "weight_group",
    "bmi_group",
    "icd10_codes",
]

MINUS_ONE_TO_NA_COLUMNS = [
    "weight_group",
    "hosp_los",
    "icu_los",
    "icu_readmit",
    "dialysis_free_days",
    "vent_free_days",
]

BMI_GROUP_MAP = {
    "L": "Underweight",
    "M": "Normal Weight",
    "P": "Overweight",
    "1": "Obesity Class 1",
    "2": "Obesity Class 2",
    "3": "Obesity Class 3",
    "X": np.nan,
    "NAN": np.nan,
}

display(
    pd.DataFrame(
        [{"final_column": column, "raw_source_columns": STANDARD_SOURCES.get(column, ["derived from filename"])} for column in FINAL_COLUMNS]
    )
)


,final_column,raw_source_columns
0,hospital_id,[derived from filename]
1,stay_id,"[PseudoID, Pseudo-ID]"
2,age_group,"[clusterAlter, ClusterAlter]"
3,sex,"[clusterGeschlecht, ClusterGeschlecht]"
4,height_group,"[clusterKoerpergroesse, ClusterKoerpergroesse]"
5,weight_group,"[clusterKoerpergewicht, ClusterKoerperGewicht]"
6,bmi_group,[BMI]
7,hosp_mortality,"[Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)]"
8,icu_mortality,[Sterblichkeit]
9,hosp_los,[Liegedauer_KH]


## Harmonization Functions

These implement the current recoding rules.


In [6]:
def pick_source_series(df: pd.DataFrame, raw_candidates: list[str], hospital: str, canonical_name: str) -> tuple[pd.Series, list[str]]:
    present = [column for column in raw_candidates if column in df.columns]
    if not present:
        return pd.Series(pd.NA, index=df.index, dtype="object"), []

    base = df[present[0]].copy()
    for other in present[1:]:
        if not series_values_equal(base, df[other]):
            raise ValueError(f"Conflicting duplicate columns for {canonical_name!r} in {hospital}: {present}")
    return base, present


def recode_sex(series: pd.Series) -> pd.Series:
    cleaned = to_clean_string(series).str.upper()
    return cleaned.replace({"W": "F"})


def recode_bmi_group(series: pd.Series) -> pd.Series:
    cleaned = to_clean_string(series).str.upper()
    return cleaned.replace(BMI_GROUP_MAP)


def derive_hosp_mortality(series: pd.Series) -> pd.Series:
    cleaned = to_clean_string(series).str.lower()
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    died_mask = cleaned.str.contains("verstorben", na=False) | cleaned.str.contains("verstoben", na=False)
    survived_mask = cleaned.str.contains("verlegt", na=False)
    result[died_mask] = 1
    result[survived_mask] = 0
    return result


def derive_icu_mortality(series: pd.Series) -> pd.Series:
    cleaned = to_clean_string(series).str.upper()
    result = pd.Series(pd.NA, index=series.index, dtype="Int64")
    result[cleaned == "ICU"] = 1
    result[cleaned.isin(["0", "KH"])] = 0
    return result


def cast_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    for column in STRING_COLUMNS:
        result[column] = to_clean_string(result[column])
    for column in NUMERIC_COLUMNS:
        result[column] = pd.to_numeric(result[column], errors="coerce").astype("Int64")
    return result


def harmonize_static_table(df: pd.DataFrame, hospital: str) -> tuple[pd.DataFrame, dict[str, list[str]]]:
    harmonized = pd.DataFrame(index=df.index)
    harmonized["hospital_id"] = hospital
    source_map = {"hospital_id": ["derived from filename"]}

    for column in ["stay_id", "sex", "age_group", "height_group", "weight_group", "bmi_group", "hosp_los", "icu_los", "icu_readmit", "dialysis_free_days", "vent_free_days", "icd10_codes"]:
        raw_series, sources = pick_source_series(df, STANDARD_SOURCES[column], hospital, column)
        harmonized[column] = raw_series
        source_map[column] = sources

    discharge_raw, sources = pick_source_series(df, STANDARD_SOURCES["hosp_mortality"], hospital, "hosp_mortality")
    harmonized["hosp_mortality"] = derive_hosp_mortality(discharge_raw)
    source_map["hosp_mortality"] = sources

    death_raw, sources = pick_source_series(df, STANDARD_SOURCES["icu_mortality"], hospital, "icu_mortality")
    harmonized["icu_mortality"] = derive_icu_mortality(death_raw)
    source_map["icu_mortality"] = sources

    harmonized["sex"] = recode_sex(harmonized["sex"])
    harmonized["bmi_group"] = recode_bmi_group(harmonized["bmi_group"])

    for column in MINUS_ONE_TO_NA_COLUMNS:
        harmonized[column] = replace_minus_one_with_na(harmonized[column])

    harmonized = cast_columns(harmonized[FINAL_COLUMNS])
    return harmonized, source_map


## Harmonized Tables

Build the aligned static tables and keep track of which raw columns were used.


In [7]:
duplicate_id_check = []
for hospital, df in raw_static_tables.items():
    if "PseudoID" in df.columns and "Pseudo-ID" in df.columns:
        duplicate_id_check.append(
            {
                "hospital": hospital,
                "PseudoID_equals_Pseudo-ID": series_values_equal(df["PseudoID"], df["Pseudo-ID"]),
            }
        )

display(pd.DataFrame(duplicate_id_check).sort_values("hospital").reset_index(drop=True))

harmonized_tables = {}
source_map_rows = []
for hospital, df in raw_static_tables.items():
    harmonized_df, source_map = harmonize_static_table(df, hospital)
    harmonized_tables[hospital] = harmonized_df
    for column in FINAL_COLUMNS:
        source_map_rows.append(
            {
                "hospital": hospital,
                "final_column": column,
                "raw_source_columns_used": source_map[column],
            }
        )

display(pd.DataFrame(source_map_rows).head(30))


,hospital,PseudoID_equals_Pseudo-ID
0,asic_UK00,True
1,asic_UK02,True
2,asic_UK03,True
3,asic_UK04,True
4,asic_UK06,True
5,asic_UK07,True
6,asic_UK08,True


,hospital,final_column,raw_source_columns_used
0,asic_UK00,hospital_id,[derived from filename]
1,asic_UK00,stay_id,"[PseudoID, Pseudo-ID]"
2,asic_UK00,age_group,[clusterAlter]
3,asic_UK00,sex,[clusterGeschlecht]
4,asic_UK00,height_group,[clusterKoerpergroesse]
5,asic_UK00,weight_group,[clusterKoerpergewicht]
6,asic_UK00,bmi_group,[BMI]
7,asic_UK00,hosp_mortality,"[Entlassgrund_(verlegt_intern,_verlegt_extern,_verstorben)]"
8,asic_UK00,icu_mortality,[Sterblichkeit]
9,asic_UK00,hosp_los,[Liegedauer_KH]


In [8]:
schema_check_rows = []
for hospital, df in harmonized_tables.items():
    schema_check_rows.append(
        {
            "hospital": hospital,
            "rows": df.shape[0],
            "final_columns": df.shape[1],
            "empty_columns": empty_columns(df),
        }
    )

display(pd.DataFrame(schema_check_rows).sort_values("hospital").reset_index(drop=True))

combined_harmonized_df = pd.concat(harmonized_tables.values(), ignore_index=True)
display(combined_harmonized_df.head())


,hospital,rows,final_columns,empty_columns
0,asic_UK00,10,15,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
1,asic_UK02,10,15,[]
2,asic_UK03,10,15,"[icu_mortality, hosp_los, icu_readmit, dialysis_free_days, vent_free_days]"
3,asic_UK04,10,15,"[dialysis_free_days, vent_free_days]"
4,asic_UK06,10,15,[]
5,asic_UK07,10,15,[]
6,asic_UK08,10,15,"[hosp_los, dialysis_free_days, vent_free_days]"


,hospital_id,stay_id,age_group,sex,height_group,weight_group,bmi_group,hosp_mortality,icu_mortality,hosp_los,icu_los,icu_readmit,dialysis_free_days,vent_free_days,icd10_codes
0,asic_UK00,372,<70,F,<180,<65,Normal Weight,1,<NA>,<NA>,4,<NA>,<NA>,<NA>,"C34,C77,C78,C79,D25,D37,D61,D69,D70,E46,E83,E87,G83,I26,I74,J40,J90,J96,K08,L89,M16,M49,M89,R00,R04,R11,R40,R52,R63,U99,Z03,Z11,Z51,Z74,Z80,Z91,Z92"
1,asic_UK00,419,<70,M,>185,76-250,Overweight,0,<NA>,<NA>,14,<NA>,<NA>,<NA>,"A09,A41,B95,B96,D62,D63,D68,E11,E87,F05,G61,G63,I10,I48,I79,J18,J91,J95,J98,K50,L03,L89,M62,M86,N08,N13,N17,N18,N39,R18,R33,R57,R63,R65,T25,T79,T87,U69,U81,U99,Z11,Z29"
2,asic_UK00,1014,<70,M,<180,76-250,Overweight,0,<NA>,<NA>,28,<NA>,<NA>,<NA>,"B97,D62,D65,E03,E87,E88,F05,I27,J12,J80.02,J96,J98,L89,N17,R13,R65,U07.1,U10,Z29,Z43"
3,asic_UK00,1204,<70,M,180-185,76-250,Overweight,0,<NA>,<NA>,4,<NA>,<NA>,<NA>,"B95,B96,D69,E11,E78,E79,G63,I21,I25,I72,J90,J96,K21,L97,L98,M14,M96,T84,U69,U99,Z11,Z92,Z98"
4,asic_UK00,1955,<70,M,180-185,76-250,Overweight,0,<NA>,<NA>,14,<NA>,<NA>,<NA>,"A41,D62,D68,F05,J15,J95,J98,R65,S01,S02,S21,S22,S25,S26,S27,S31,S32,S36,S37,S39,S42,S51,S52,S91,S93,T79,U69,U99,V99,Z11,Z93"


## Recoding Sanity Checks

Quick summaries for the recoded categorical and outcome fields.


In [9]:
for column in ["sex", "bmi_group", "hosp_mortality", "icu_mortality"]:
    display(Markdown(f"### {column}"))
    summary = (
        combined_harmonized_df.groupby("hospital_id", dropna=False)[column]
        .value_counts(dropna=False)
        .unstack(fill_value=0)
        .reset_index()
    )
    display(summary)


### sex

sex,hospital_id,F,M
0,asic_UK00,3,7
1,asic_UK02,4,6
2,asic_UK03,4,6
3,asic_UK04,4,6
4,asic_UK06,4,6
5,asic_UK07,6,4
6,asic_UK08,2,8


### bmi_group

bmi_group,hospital_id,Normal Weight,Obesity Class 1,Obesity Class 2,Obesity Class 3,Overweight,Underweight,<NA>
0,asic_UK00,1,0,0,1,8,0,0
1,asic_UK02,2,1,0,2,4,1,0
2,asic_UK03,2,0,0,0,7,1,0
3,asic_UK04,3,1,0,0,4,0,2
4,asic_UK06,3,2,1,0,4,0,0
5,asic_UK07,2,3,1,1,2,1,0
6,asic_UK08,2,4,0,0,3,1,0


### hosp_mortality

hosp_mortality,hospital_id,0,1
0,asic_UK00,7,3
1,asic_UK02,7,3
2,asic_UK03,3,7
3,asic_UK04,9,1
4,asic_UK06,3,7
5,asic_UK07,8,2
6,asic_UK08,9,1


### icu_mortality

icu_mortality,hospital_id,0,1,<NA>
0,asic_UK00,0,0,10
1,asic_UK02,7,3,0
2,asic_UK03,0,0,10
3,asic_UK04,10,0,0
4,asic_UK06,3,7,0
5,asic_UK07,8,2,0
6,asic_UK08,9,1,0


## Final Value Preview By Hospital

Preview the harmonized columns in their grouped final order.


In [10]:
for hospital in sorted(harmonized_tables):
    display(Markdown(f"### {hospital}"))
    display(value_preview(harmonized_tables[hospital], max_values=8))


### asic_UK00

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10,1,[asic_UK00]
1,stay_id,Int64,10,10,"[372, 419, 1014, 1204, 1955, 2203, 2353, 2578, ...]"
2,age_group,string,10,3,"[<70, 70-79, 80-130]"
3,sex,string,10,2,"[F, M]"
4,height_group,string,10,3,"[<180, >185, 180-185]"
5,weight_group,string,10,2,"[<65, 76-250]"
6,bmi_group,string,10,3,"[Normal Weight, Overweight, Obesity Class 3]"
7,hosp_mortality,Int64,10,2,"[1, 0]"
8,icu_mortality,Int64,0,0,[]
9,hosp_los,Int64,0,0,[]


### asic_UK02

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10,1,[asic_UK02]
1,stay_id,Int64,10,10,"[237, 334, 403, 440, 478, 506, 611, 723, ...]"
2,age_group,string,10,3,"[80-130, 70-79, <70]"
3,sex,string,10,2,"[M, F]"
4,height_group,string,10,2,"[<180, 180-185]"
5,weight_group,string,10,3,"[<65, 76-250, 65-75]"
6,bmi_group,string,10,5,"[Underweight, Obesity Class 3, Overweight, Normal Weight, Obesity Class 1]"
7,hosp_mortality,Int64,10,2,"[0, 1]"
8,icu_mortality,Int64,10,2,"[0, 1]"
9,hosp_los,Int64,10,10,"[17, 37, 3, 15, 12, 14, 30, 61, ...]"


### asic_UK03

...

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10,1,[asic_UK06]
1,stay_id,Int64,10,10,"[237, 334, 403, 440, 478, 506, 578, 604, ...]"
2,age_group,string,10,3,"[<70, 80-130, 70-79]"
3,sex,string,10,2,"[F, M]"
4,height_group,string,10,2,"[<180, 180-185]"
5,weight_group,string,10,3,"[76-250, 65-75, <65]"
6,bmi_group,string,10,4,"[Obesity Class 2, Obesity Class 1, Overweight, Normal Weight]"
7,hosp_mortality,Int64,10,2,"[0, 1]"
8,icu_mortality,Int64,10,2,"[0, 1]"
9,hosp_los,Int64,10,8,"[18, 23, 3, 14, 12, 11, 30, 22]"


### asic_UK07

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10,1,[asic_UK07]
1,stay_id,Int64,10,10,"[372, 419, 1014, 1204, 1221, 1258, 1304, 1934, ...]"
2,age_group,string,10,2,"[<70, 70-79]"
3,sex,string,10,2,"[F, M]"
4,height_group,string,10,2,"[<180, 180-185]"
5,weight_group,string,10,3,"[76-250, 65-75, <65]"
6,bmi_group,string,10,6,"[Obesity Class 1, Obesity Class 2, Normal Weight, Obesity Class 3, Overweight, Underweight]"
7,hosp_mortality,Int64,10,2,"[0, 1]"
8,icu_mortality,Int64,10,2,"[0, 1]"
9,hosp_los,Int64,10,10,"[41, 67, 28, 33, 43, 31, 35, 34, ...]"


### asic_UK08

,column,dtype,non_null_count,n_unique_non_null,example_values
0,hospital_id,string,10,1,[asic_UK08]
1,stay_id,Int64,10,10,"[237, 403, 478, 611, 723, 2074, 2142, 2316, ...]"
2,age_group,string,10,2,"[<70, 70-79]"
3,sex,string,10,2,"[F, M]"
4,height_group,string,10,2,"[<180, 180-185]"
5,weight_group,string,10,3,"[65-75, 76-250, <65]"
6,bmi_group,string,10,4,"[Overweight, Obesity Class 1, Normal Weight, Underweight]"
7,hosp_mortality,Int64,10,2,"[0, 1]"
8,icu_mortality,Int64,10,2,"[0, 1]"
9,hosp_los,Int64,0,0,[]
